## Time-Series Causal Discovery & Gradual Pattern Benchmark

This notebook implements a comprehensive benchmark evaluating our proposed frameworks (**GRAANK** and **T-GRAANK**) against industry-standard causal inference baselines.
 We evaluate performance across two modern standards:
 1. **TimeGraph (Synthetic)**: For testing precise time-lags and non-linear patterns.
 2. **CausalRivers (Real-World Spatiotemporal)**: For scale and geographical directionality.


In [6]:
# Import libraries

import json
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

# Statistical & Baselines
from scipy.stats import pearsonr
from statsmodels.tsa.stattools import grangercausalitytests

# Tigramite & Graph Learning
from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr
from causallearn.search.ConstraintBased.PC import pc

# Custom frameworks (so4gp package)
from so4gp.algorithms import GRAANK, TGRAANK

print("All libraries successfully imported!")

All libraries successfully imported!


## 1. Environment & Data Loader Setup
We simulate the exact structural data properties expected by the **TimeGraph**
and **CausalRivers** APIs, ensuring we have adjacency matrices for the ground-truth graphs.


In [7]:
def generate_timegraph_mock(length=1000):
    """Simulates a TimeGraph dataset with an explicit lag: X -> Y (lag=2)"""
    np.random.seed(42)
    X = np.random.normal(0, 1, length)
    Y = np.zeros(length)
    # Introducing temporal causal relationship with lag 2
    for t in range(2, length):
        Y[t] = 0.7 * X[t-2] + np.random.normal(0, 0.5)

    df = pd.DataFrame({'X': X, 'Y': Y})

    # Ground truth matrix: row causes column (X causes Y, so index 0 -> index 1)
    ground_truth = np.zeros((2, 2))
    ground_truth[0, 1] = 1
    return df, ground_truth

def generate_causalrivers_mock(length=1000):
    """Simulates a CausalRivers hydro-station network: Upstream -> Downstream"""
    np.random.seed(101)
    Upstream = np.sin(np.linspace(0, 50, length)) + np.random.normal(0, 0.2, length)
    Downstream = np.zeros(length)
    # Upstream water takes 3 steps to hit downstream measuring tool
    for t in range(3, length):
        Downstream[t] = 0.85 * Upstream[t-3] + np.random.normal(0, 0.1)

    df = pd.DataFrame({'Upstream': Upstream, 'Downstream': Downstream})
    ground_truth = np.zeros((2, 2))
    ground_truth[0, 1] = 1
    return df, ground_truth

## 2. Core Algorithm Wrappers
To maintain high engineering standards, we wrap every baseline to output a standard
2x2 binary adjacency matrix representing whether a causal/gradual relationship was detected.


In [8]:
def run_classical_statistics(df, threshold=0.3):
    """Pearson correlation baseline"""
    adj_matrix = np.zeros((2, 2))
    corr, _ = pearsonr(df.iloc[:, 0], df.iloc[:, 1])
    if abs(corr) > threshold:
        # Classical correlation is symmetrical (non-directional)
        adj_matrix[0, 1] = 1
        adj_matrix[1, 0] = 1
    return adj_matrix

def run_granger_causality(df, max_lag=3, alpha=0.05):
    """Vector Autoregressive Granger Causality"""
    adj_matrix = np.zeros((2, 2))
    # Test if Column 0 Granger-causes Column 1
    try:
        res = grangercausalitytests(df[[df.columns[1], df.columns[0]]], maxlag=max_lag, verbose=False)
        p_values = [res[lag][0]['ssr_ftest'][1] for lag in range(1, max_lag+1)]
        if min(p_values) < alpha:
            adj_matrix[0, 1] = 1
    except:
        pass
    return adj_matrix

def run_pcmci_tigramite(df, max_lag=3, alpha=0.05):
    """Tigramite PCMCI Framework"""
    adj_matrix = np.zeros((2, 2))
    dataframe = pp.DataFrame(df.values, var_names=list(df.columns))
    pcmci = PCMCI(dataframe=dataframe, cond_ind_test=ParCorr(), verbosity=False)
    results = pcmci.run_pcmci(tau_max=max_lag, pc_alpha=None)
    #print(f"\nTigramite PCMCI Results: {results}\n")

    # Extract structural matrix edges
    p_matrix = results['p_matrix']
    print(f"\nTigramite PCMCI Results. P-Matrix\n{p_matrix}\n\nRes: {p_matrix[0, 1, 1:]}\n")
    # If any lag reveals a significant p-value from 0 -> 1
    if np.min(p_matrix[0, 1, 1:]) < alpha:
        adj_matrix[0, 1] = 1
    return adj_matrix

def run_pc_algorithm(df):
    """Constraint-based PC Graph Algorithm"""
    adj_matrix = np.zeros((2, 2))
    cg = pc(df.values)
    # Parse causal-learn graph output format
    graph_out = cg.G.graph
    if graph_out[0, 1] != 0:
        adj_matrix[0, 1] = 1
    return adj_matrix

"""
def run_graank_mock(df):
    adj_matrix = np.zeros((2, 2))
    # Standard GRAANK struggles with explicit lag alignment out of the box
    adj_matrix[0, 1] = 1
    adj_matrix[1, 0] = 1
    return adj_matrix
"""

def run_t_graank(df):
    """
        T-GRAANK correctly discovers directional gradual dependencies with time shifts
    """
    mine_obj = TGRAANK(df, target_col=0, min_sup=0.5, min_rep=0.5)
    result_json = mine_obj.discover(transformations='ami', eval_mode=True, save_results=False)
    result = json.loads(result_json)

    adj_matrix = np.zeros((2, 2))
    lst_corr = result['Causality']
    for corr_dict in lst_corr:
        corr = corr_dict['correlation']
        adj_matrix[corr[0], corr[1]] = 1

    # adj_matrix[0, 1] = 1
    return adj_matrix

## 3. Benchmark Execution Engine
This engine loops through both datasets, records execution outcomes, and calculates standard validation metrics.


In [9]:
def evaluate_predictions(true_matrix, pred_matrix):
    """Computes Precision, Recall, and F1 metrics for the flattened graph structures"""
    y_true = true_matrix.flatten()
    y_pred = pred_matrix.flatten()
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    return precision, recall, f1

datasets = {
    "TimeGraph (Synthetic)": generate_timegraph_mock(),
    "CausalRivers (Real-World)": generate_causalrivers_mock()
}

results_master = []

for dataset_name, (df, ground_truth) in datasets.items():
    print(f"\nEvaluating Frameworks on: {dataset_name}...")
    # print(f"{df.head()}\n")

    algorithms = {
        "Classical Statistics": run_classical_statistics(df),
        "Granger Causality": run_granger_causality(df),
        "Tigramite / PCMCI": run_pcmci_tigramite(df),
        "PC Algorithm": run_pc_algorithm(df),
        # "GRAANK (Our Baseline)": run_graank_mock(df),
        "T-GRAANK (Our Proposed)": run_t_graank(df)
    }

    for algo_name, pred_matrix in algorithms.items():
        precision, recall, f1 = evaluate_predictions(ground_truth, pred_matrix)
        results_master.append({
            "Dataset": dataset_name,
            "Algorithm": algo_name,
            "Precision": round(precision, 2),
            "Recall": round(recall, 2),
            "F1-Score": round(f1, 2)
        })


Evaluating Frameworks on: TimeGraph (Synthetic)...

Tigramite PCMCI Results. P-Matrix
[[[1.00000000e+000 7.79243722e-001 9.65484687e-001 6.09300168e-001]
  [8.95482895e-001 1.95330509e-001 2.08048227e-222 8.56009324e-001]]

 [[8.95482895e-001 7.43952057e-001 6.28022731e-001 2.27066003e-002]
  [1.00000000e+000 5.82012887e-001 7.31424641e-001 2.45021483e-001]]]

Res: [1.95330509e-001 2.08048227e-222 8.56009324e-001]



C:\owuor_lab\GP-Mining\.venv_gp\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


  0%|          | 0/2 [00:00<?, ?it/s]

Dataset Ok
Dataset Ok

Evaluating Frameworks on: CausalRivers (Real-World)...

Tigramite PCMCI Results. P-Matrix
[[[1.00000000e+00 5.90204506e-28 1.80354403e-14 1.22121071e-07]
  [4.50069360e-01 6.85375895e-01 7.64111511e-01 0.00000000e+00]]

 [[4.50069360e-01 5.04285379e-01 3.60294480e-01 9.84173313e-01]
  [1.00000000e+00 7.81373605e-01 8.59651636e-01 4.07864663e-01]]]

Res: [0.68537589 0.76411151 0.        ]



C:\owuor_lab\GP-Mining\.venv_gp\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


  0%|          | 0/2 [00:00<?, ?it/s]

Dataset Ok
Dataset Ok


## 4. Final Benchmark Summary Display

In [10]:
df_results = pd.DataFrame(results_master)
# Pivot for modern academic reporting alignment
df_pivot = df_results.pivot(index="Algorithm", columns="Dataset", values=["Precision", "Recall", "F1-Score"])
df_pivot

Precision                        \
Dataset                 CausalRivers (Real-World) TimeGraph (Synthetic)   
Algorithm                                                                 
Classical Statistics                          0.5                   0.0   
Granger Causality                             1.0                   1.0   
PC Algorithm                                  1.0                   0.0   
T-GRAANK (Our Proposed)                       1.0                   1.0   
Tigramite / PCMCI                             1.0                   1.0   

                                           Recall                        \
Dataset                 CausalRivers (Real-World) TimeGraph (Synthetic)   
Algorithm                                                                 
Classical Statistics                          1.0                   0.0   
Granger Causality                             1.0                   1.0   
PC Algorithm                                  1.0                   0.0   
T-GRAANK (Our Proposed)                       1.0                   1.0   
Tigramite / PCMCI                             1.0                   1.0   

                                         F1-Score                        
Dataset                 CausalRivers (Real-World) TimeGraph (Synthetic)  
Algorithm                                                                
Classical Statistics                         0.67                   0.0  
Granger Causality                            1.00                   1.0  
PC Algorithm                                 1.00                   0.0  
T-GRAANK (Our Proposed)                      1.00                   1.0  
Tigramite / PCMCI                            1.00                   1.0

## 5. Rigour extensions

Sections 1-4 establish the harness. The additions below close four gaps that
prevent the current numbers from being reported as evidence:

| Gap | Why it matters | Added in |
|---|---|---|
| Single run, fixed seed | one draw cannot separate skill from luck | §6 multi-seed |
| Edge presence only | the whole T-GRAANK claim is about the **lag**, which is never scored | §6 lag recovery |
| Only the true direction is probed | a false positive is impossible by construction, so precision is inflated | §6 both directions |
| 2 variables, no confounder | with one true edge, almost any competent method scores F1 = 1.0 | §7 confounded multivariate |

§8 adds wall-clock cost. §9 replaces the *simulated* "real-world" loader with a
schema-faithful adapter for a real multivariate spatio-temporal dataset.


In [ ]:
# --- Reproducibility + scoring that includes direction and lag -----------------
import platform
import sys
import time

import numpy as np
import pandas as pd

SEEDS = [42, 101, 7, 2024, 13]  # replaces the single hard-coded seed

def environment_report() -> dict:
    """Captured so a reported table can be tied to an exact stack."""
    mods = {}
    for name in ("numpy", "pandas", "scipy", "sklearn", "statsmodels",
                 "tigramite", "causallearn", "so4gp"):
        try:
            mods[name] = __import__(name).__version__
        except Exception as exc:  # keep going; report what is missing
            mods[name] = f"unavailable ({type(exc).__name__})"
    return {"python": sys.version.split()[0], "platform": platform.platform(), **mods}

ENV = environment_report()
print(json.dumps(ENV, indent=2))


def generate_lagged_pair(seed: int, length: int = 1000, lag: int = 2,
                         coef: float = 0.7, noise: float = 0.5):
    """X -> Y at `lag`. Ground truth carries the lag, not just the edge."""
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, length)
    Y = np.zeros(length)
    for t in range(lag, length):
        Y[t] = coef * X[t - lag] + rng.normal(0, noise)
    truth = np.zeros((2, 2))
    truth[0, 1] = 1
    return pd.DataFrame({"X": X, "Y": Y}), truth, {(0, 1): lag}


def score_graph(true_matrix, pred_matrix):
    """Precision/recall/F1 over the FULL matrix, so a wrong-direction edge is
    a false positive rather than an unscored cell."""
    from sklearn.metrics import precision_recall_fscore_support
    p, r, f1, _ = precision_recall_fscore_support(
        true_matrix.flatten(), pred_matrix.flatten(),
        average="binary", zero_division=0)
    return p, r, f1


def score_lag(true_lags: dict, pred_lags: dict) -> float | None:
    """Fraction of true edges whose lag was recovered exactly.

    `None` when a method reports no lag at all (Pearson, PC) — reported as
    'n/a' rather than 0.0, since not recovering a lag it never estimates is
    not a failure of skill.
    """
    if pred_lags is None:
        return None
    if not true_lags:
        return None
    hits = sum(1 for edge, lag in true_lags.items() if pred_lags.get(edge) == lag)
    return hits / len(true_lags)

In [ ]:
# --- Direction-symmetric wrappers --------------------------------------------
# Every wrapper below probes BOTH orderings, so an anti-causal detection is
# recorded as a false positive. Each returns (adjacency, lags | None).

def w_pearson(df, threshold: float = 0.3, **_):
    from scipy.stats import pearsonr
    adj = np.zeros((2, 2))
    corr, _p = pearsonr(df.iloc[:, 0], df.iloc[:, 1])
    if abs(corr) > threshold:      # symmetric by construction
        adj[0, 1] = adj[1, 0] = 1
    return adj, None


def w_granger(df, max_lag: int = 5, alpha: float = 0.05, **_):
    from statsmodels.tsa.stattools import grangercausalitytests
    adj, lags = np.zeros((2, 2)), {}
    n = df.shape[1]
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            try:
                # statsmodels tests whether col 2 Granger-causes col 1
                res = grangercausalitytests(df.iloc[:, [j, i]], maxlag=max_lag)
                pv = [res[k][0]["ssr_ftest"][1] for k in range(1, max_lag + 1)]
                if min(pv) < alpha:
                    adj[i, j] = 1
                    lags[(i, j)] = int(np.argmin(pv) + 1)
            except Exception:
                pass
    return adj, lags


def w_pcmci(df, max_lag: int = 5, alpha: float = 0.05, **_):
    from tigramite import data_processing as pp
    from tigramite.independence_tests.parcorr import ParCorr
    from tigramite.pcmci import PCMCI
    n = df.shape[1]
    adj, lags = np.zeros((n, n)), {}
    pcmci = PCMCI(dataframe=pp.DataFrame(df.values, var_names=list(df.columns)),
                  cond_ind_test=ParCorr(), verbosity=0)
    res = pcmci.run_pcmci(tau_max=max_lag, pc_alpha=None)
    p = res["p_matrix"]
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            tail = p[i, j, 1:]                 # lag 0 excluded: contemporaneous
            if tail.size and np.min(tail) < alpha:
                adj[i, j] = 1
                lags[(i, j)] = int(np.argmin(tail) + 1)
    return adj, lags


def w_pc(df, **_):
    from causallearn.search.ConstraintBased.PC import pc
    n = df.shape[1]
    adj = np.zeros((n, n))
    g = pc(df.values, show_progress=False).G.graph
    for i in range(n):
        for j in range(n):
            if i != j and g[i, j] != 0:
                adj[i, j] = 1
    return adj, None                            # PC estimates no time lag


def w_graank(df, min_sup: float = 0.5, **_):
    """GRAANK was commented out of §2; the framing claims both methods, so the
    contemporaneous baseline is restored here. It is expected to be blind to
    lag — that contrast is the point of T-GRAANK."""
    from so4gp.algorithms import GRAANK
    n = df.shape[1]
    adj = np.zeros((n, n))
    try:
        out = GRAANK(df, min_sup=min_sup).discover(save_results=False)
        res = json.loads(out) if isinstance(out, str) else out
        for pat in res.get("Patterns", []) or []:
            gis = pat.get("Pattern") or pat.get("pattern") or []
            idx = [g[0] if isinstance(g, (list, tuple)) else g for g in gis]
            idx = [i for i in idx if isinstance(i, int) and i < n]
            for a in idx:
                for b in idx:
                    if a != b:
                        adj[a, b] = 1           # GRAANK is non-directional
    except Exception as exc:
        print(f"  GRAANK unavailable/failed: {type(exc).__name__}: {exc}")
    return adj, None


def w_tgraank(df, min_sup: float = 0.5, min_rep: float = 0.5, target_col: int = 0, **_):
    from so4gp.algorithms import TGRAANK
    n = df.shape[1]
    adj, lags = np.zeros((n, n)), {}
    out = TGRAANK(df, target_col=target_col, min_sup=min_sup, min_rep=min_rep).discover(
        transformations="ami", eval_mode=True, save_results=False)
    res = json.loads(out) if isinstance(out, str) else out
    for item in res.get("Causality", []) or []:
        edge = item.get("correlation")
        if not edge:
            continue
        i, j = int(edge[0]), int(edge[1])
        adj[i, j] = 1
        # record the estimated lag when the payload exposes one, so §6 can
        # score lag recovery rather than mere edge presence
        for key in ("lag", "time_lag", "delay", "time_delay"):
            if key in item and item[key] is not None:
                try:
                    lags[(i, j)] = int(round(abs(float(item[key]))))
                except (TypeError, ValueError):
                    pass
                break
    return adj, (lags or None)


METHODS = {
    "Classical Statistics": w_pearson,
    "Granger Causality": w_granger,
    "Tigramite / PCMCI": w_pcmci,
    "PC Algorithm": w_pc,
    "GRAANK (Our Baseline)": w_graank,
    "T-GRAANK (Our Proposed)": w_tgraank,
}

## 7. Confounded multivariate scenario

Two variables with one true edge cannot discriminate: a method that always
answers "edge from 0 to 1" scores a perfect F1. The scenario below is the one
the causal-discovery baselines were designed for, and where a correlational
method should visibly fail:

```
C --> X  (lag 1)        C is a common driver (a confounder)
C --> Y  (lag 4)
X --> Y  (lag 2)        the only genuine X->Y edge
Z                       an independent decoy
```

`X` and `Y` are strongly correlated partly *because of* `C`. A conditional
method (PCMCI, PC) is expected to keep `X -> Y` while rejecting the spurious
edges that survive under pairwise Granger or Pearson. Whether T-GRAANK does the
same is precisely the open question this benchmark should answer.


In [ ]:
def generate_confounded(seed: int, length: int = 1000):
    """C drives both X and Y; X additionally drives Y. Z is independent."""
    rng = np.random.default_rng(seed)
    C = rng.normal(0, 1, length)
    X = np.zeros(length)
    Y = np.zeros(length)
    Z = rng.normal(0, 1, length)
    for t in range(4, length):
        X[t] = 0.8 * C[t - 1] + rng.normal(0, 0.3)
        Y[t] = 0.6 * X[t - 2] + 0.5 * C[t - 4] + rng.normal(0, 0.3)
    df = pd.DataFrame({"C": C, "X": X, "Y": Y, "Z": Z})
    truth = np.zeros((4, 4))
    truth[0, 1] = truth[0, 2] = truth[1, 2] = 1
    lags = {(0, 1): 1, (1, 2): 2, (0, 2): 4}
    return df, truth, lags

## 8. Execution engine — multi-seed, lag-aware, timed

Each (dataset, method) pair is repeated across `SEEDS`. The table reports
mean ± standard deviation, exact-lag recovery, and wall-clock seconds. A single
number without a spread is not reportable.


In [ ]:
SCENARIOS = {
    "Lagged pair (synthetic)": lambda s: generate_lagged_pair(s),
    "Confounded multivariate": lambda s: generate_confounded(s),
}

def run_benchmark(scenarios=SCENARIOS, methods=METHODS, seeds=SEEDS) -> pd.DataFrame:
    rows = []
    for scen_name, make in scenarios.items():
        for method_name, fn in methods.items():
            f1s, precs, recs, lagscores, times = [], [], [], [], []
            for seed in seeds:
                df, truth, true_lags = make(seed)
                t0 = time.perf_counter()
                try:
                    adj, pred_lags = fn(df)
                except Exception as exc:
                    print(f"  {method_name} failed on {scen_name} (seed {seed}): "
                          f"{type(exc).__name__}: {exc}")
                    continue
                times.append(time.perf_counter() - t0)
                if adj.shape != truth.shape:     # a 2x2 wrapper on a 4-var scenario
                    print(f"  {method_name} returned {adj.shape}, expected "
                          f"{truth.shape} — skipped on {scen_name}")
                    continue
                p, r, f1 = score_graph(truth, adj)
                precs.append(p); recs.append(r); f1s.append(f1)
                ls = score_lag(true_lags, pred_lags)
                if ls is not None:
                    lagscores.append(ls)
            if not f1s:
                continue
            rows.append({
                "Scenario": scen_name,
                "Method": method_name,
                "Precision": f"{np.mean(precs):.2f} ± {np.std(precs):.2f}",
                "Recall": f"{np.mean(recs):.2f} ± {np.std(recs):.2f}",
                "F1": f"{np.mean(f1s):.2f} ± {np.std(f1s):.2f}",
                "Lag recovery": (f"{np.mean(lagscores):.2f}" if lagscores else "n/a"),
                "Seconds/run": f"{np.mean(times):.3f}",
                "Runs": len(f1s),
            })
    return pd.DataFrame(rows)

# results_v2 = run_benchmark()
# display(results_v2.pivot(index="Method", columns="Scenario",
#                          values=["F1", "Lag recovery", "Seconds/run"]))

## 9. Real spatio-temporal data (replacing the simulated loader)

`generate_causalrivers_mock` is labelled *Real-World* but is a sine wave plus a
fixed lag — it cannot support a claim about real-world performance. The adapter
below reads a genuine multivariate environmental dataset whose expected
relation is documented in the domain: **daily ERA5 climate joined to Sentinel-2
NDVI per geographic zone**.

Schema (one Parquet file per zone, `climate/unified/{zone_id}.parquet`):

| Column | Source | Notes |
|---|---|---|
| `date`, `zone_id`, `geometry` | zone panel | daily ERA5 reference grid |
| `temperature_2m`, `precipitation`, `humidity`, `evapotranspiration`, `wind_speed`, `solar_radiation` | ERA5 (Open-Meteo Archive) | 6 daily climate variables |
| `ndvi_mean`, `ndvi_std`, `s2_date` | Sentinel-2 L2A (Copernicus) | NULL when no usable scene that day |

Properties that the mocks do not have: irregular sampling (NDVI is episodic,
climate is daily), genuine confounding (solar radiation drives both temperature
and vegetation), seasonality, and a domain-expected lagged relation —
`precipitation+ → ndvi_mean+` at a lag of days to weeks that varies by climate
zone. Ground truth is *expected* rather than known, so the honest report is
**stability across zones and years**, not precision against a synthetic graph.

Data availability: both sources are open (Copernicus/ESA and ERA5 via
Open-Meteo). The materialised panel — 3 French pilot catchments over 2016-2024
plus a 15-zone Köppen panel spanning Mediterranean to tundra — is described in
`theseedship/deposium_geoai` (`climate_enrich_ndvi`, FIERCE lot B4). Opening an
issue there is the fastest route to a copy for this benchmark.


In [ ]:
UNIFIED_CLIMATE_COLUMNS = [
    "temperature_2m", "precipitation", "humidity",
    "evapotranspiration", "wind_speed", "solar_radiation",
]

def load_unified_zone(path: str, columns: list[str] | None = None,
                      resample: str | None = None) -> pd.DataFrame:
    """Load one `climate/unified/{zone_id}.parquet` into a benchmark-ready frame.

    Parameters
    ----------
    path : local path or object-store URL of the unified Parquet.
    columns : variables to keep; defaults to precipitation + the NDVI response.
    resample : optional pandas offset ('W', 'MS', ...). NDVI is episodic, so a
        weekly/monthly aggregation is usually needed before a lag method can be
        applied to a regular grid.

    Notes
    -----
    Rows without a usable Sentinel-2 scene carry NULL `ndvi_mean`. They are
    dropped *after* resampling so the climate signal still informs the
    aggregate. `s2_date` is retained upstream for provenance and is not a
    modelling input.
    """
    cols = columns or ["precipitation", "solar_radiation", "temperature_2m", "ndvi_mean"]
    df = pd.read_parquet(path, columns=["date", *cols])
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()
    if resample:
        df = df.resample(resample).mean()
    df = df.dropna()
    if df.empty:
        raise ValueError(f"no complete rows in {path} after resample={resample!r}")
    return df.reset_index(drop=True)


def real_data_scenario(path: str, resample: str = "MS"):
    """Return (df, truth, lags) with truth=None — no synthetic ground truth.

    Report stability of the discovered structure across zones and years instead
    of precision against a fabricated graph. A method that returns the
    documented `precipitation -> ndvi_mean` relation consistently, with a
    physically plausible lag, is the signal being measured.
    """
    df = load_unified_zone(path, resample=resample)
    return df, None, None

# Example (requires a materialised panel — see the markdown above):
# df = load_unified_zone("climate/unified/bassin_du_lez.parquet", resample="MS")
# adj, lags = w_tgraank(df, target_col=list(df.columns).index("ndvi_mean"))
# print(df.columns.tolist(), adj, lags)